# Pump Vibration Anomaly Detector — Colab T4 Training

**Before running:** Runtime → Change runtime type → T4 GPU

Uses the **CWRU Bearing Dataset** — real vibration recordings from a motor-driven mechanical system. Normal (baseline) recordings are used for training. Fault recordings are used only for threshold evaluation.

Checkpoints are mirrored to Google Drive every epoch so training survives disconnects.

In [ ]:
# Cell 1: Verify GPU and mount Drive
import torch
assert torch.cuda.is_available(), 'No GPU found — set Runtime > Change runtime type > T4'
print(f'GPU : {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/pump-anomaly/checkpoints'
import os
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'Drive mounted. Checkpoints -> {DRIVE_DIR}')

In [ ]:
# Cell 2: Clone repo and install dependencies
!git clone https://github.com/YOUR_USERNAME/pump-anomaly-detector.git
%cd pump-anomaly-detector
!pip install -q -r requirements.txt
!pip install -q scipy mat73
print('Dependencies installed.')

In [ ]:
# Cell 3: Download CWRU Bearing Dataset (real vibration data)
#
# Case Western Reserve University Bearing Data Centre
# https://engineering.case.edu/bearingdatacenter
#
# We download:
#   Normal baseline recordings    -> training data (no faults)
#   Ball fault / inner race fault  -> evaluation only (never seen during training)
#
# Each .mat file contains drive-end (DE) and fan-end (FE) accelerometer channels
# sampled at 12 kHz or 48 kHz. We use the 12 kHz drive-end channel.

import urllib.request, os, scipy.io, numpy as np

os.makedirs('data/raw', exist_ok=True)

FILES = {
    # (filename, url, label)  label 0=normal, 1=fault
    'normal_0.mat'  : ('https://engineering.case.edu/sites/default/files/97.mat',   0),
    'normal_1.mat'  : ('https://engineering.case.edu/sites/default/files/98.mat',   0),
    'normal_2.mat'  : ('https://engineering.case.edu/sites/default/files/99.mat',   0),
    'ball_007.mat'  : ('https://engineering.case.edu/sites/default/files/105.mat',  1),
    'ball_014.mat'  : ('https://engineering.case.edu/sites/default/files/169.mat',  1),
    'inner_007.mat' : ('https://engineering.case.edu/sites/default/files/109.mat',  1),
    'inner_014.mat' : ('https://engineering.case.edu/sites/default/files/174.mat',  1),
}

def extract_de_channel(mat_path):
    """Extract drive-end accelerometer signal from a CWRU .mat file."""
    mat = scipy.io.loadmat(mat_path)
    # The DE channel key looks like 'X097_DE_time', 'X105_DE_time', etc.
    de_key = next(k for k in mat if 'DE_time' in k)
    return mat[de_key].ravel().astype(np.float32)

normal_segments, fault_segments = [], []

for fname, (url, label) in FILES.items():
    fpath = f'data/raw/{fname}'
    if not os.path.exists(fpath):
        print(f'Downloading {fname} ...', end=' ')
        urllib.request.urlretrieve(url, fpath)
        print('done')
    else:
        print(f'{fname} already exists, skipping download')

    sig = extract_de_channel(fpath)
    if label == 0:
        normal_segments.append(sig)
    else:
        fault_segments.append(sig)

normal_signal = np.concatenate(normal_segments)
fault_signal  = np.concatenate(fault_segments)

np.save('data/raw/normal_signal.npy', normal_signal)
np.save('data/raw/fault_signal.npy',  fault_signal)

print(f'\nNormal signal : {len(normal_signal):,} samples  ({len(normal_signal)/12000:.1f} s at 12 kHz)')
print(f'Fault signal  : {len(fault_signal):,} samples  ({len(fault_signal)/12000:.1f} s at 12 kHz)')
print('Real vibration data ready.')

In [ ]:
# Cell 3b: Quick plot of normal vs fault vibration signal (sanity check)
import matplotlib.pyplot as plt
import numpy as np

normal_signal = np.load('data/raw/normal_signal.npy')
fault_signal  = np.load('data/raw/fault_signal.npy')

fig, axes = plt.subplots(2, 1, figsize=(14, 5), sharex=True)
t = np.arange(2048) / 12000 * 1000  # ms
axes[0].plot(t, normal_signal[:2048], lw=0.7, color='steelblue')
axes[0].set_title('Normal bearing vibration (CWRU drive-end, 12 kHz)')
axes[0].set_ylabel('Acceleration (g)')
axes[1].plot(t, fault_signal[:2048], lw=0.7, color='crimson')
axes[1].set_title('Faulty bearing vibration (ball fault, 0.007 inch defect)')
axes[1].set_ylabel('Acceleration (g)')
axes[1].set_xlabel('Time (ms)')
plt.tight_layout()
plt.savefig('data_preview.png', dpi=150)
plt.show()

In [ ]:
# Cell 4: Restore checkpoint from previous session (skip on very first run)
import shutil, os
os.makedirs('models/saved', exist_ok=True)
drive_last = f'{DRIVE_DIR}/checkpoint_last.pt'
drive_best = f'{DRIVE_DIR}/checkpoint_best.pt'
if os.path.exists(drive_last) and os.path.exists(drive_best):
    shutil.copy2(drive_last, 'models/saved/checkpoint_last.pt')
    shutil.copy2(drive_best, 'models/saved/checkpoint_best.pt')
    print('Checkpoint restored from Drive — will resume from last epoch')
    if os.path.exists(f'{DRIVE_DIR}/history.json'):
        shutil.copy2(f'{DRIVE_DIR}/history.json', 'models/saved/history.json')
else:
    print('No previous checkpoint — starting fresh')

In [ ]:
# Cell 5: Train on real CWRU normal data
#
# The autoencoder is trained ONLY on normal bearing vibration.
# Fault data is never seen during training.
# High reconstruction error at inference = anomaly.
#
# History is written to models/saved/history.json after every epoch
# so Cell 6 works even if Colab cuts the session short.

import sys, numpy as np, torch
sys.path.insert(0, '.')
from models.autoencoder import PumpAutoencoder
from utils.dataset import VibrationDataset, build_dataloaders
from utils.trainer import Trainer
import shutil, os

WINDOW   = 512
BATCH    = 128
ACCUM    = 4       # effective batch = 512
LR       = 1e-3
EPOCHS   = 100
PATIENCE = 15

normal_signal = np.load('data/raw/normal_signal.npy')
print(f'Normal signal: {len(normal_signal):,} samples')

train_loader, val_loader = build_dataloaders(
    normal_signal,
    window_size=WINDOW,
    stride=WINDOW // 4,
    batch_size=BATCH,
)
print(f'Train batches: {len(train_loader)} | Val batches: {len(val_loader)}')

model = PumpAutoencoder(input_length=WINDOW, latent_dim=32)

trainer = Trainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    lr=LR,
    checkpoint_dir='models/saved',
    accumulation_steps=ACCUM,
    use_amp=True,
)

# Hook: mirror checkpoints + history to Drive after every save
_orig_save = trainer.save_checkpoint
def _save_and_mirror(epoch, val_loss, tag='last'):
    _orig_save(epoch, val_loss, tag)
    for fname in ['checkpoint_last.pt', 'checkpoint_best.pt', 'history.json']:
        src = f'models/saved/{fname}'
        if os.path.exists(src):
            shutil.copy2(src, f'{DRIVE_DIR}/{fname}')
trainer.save_checkpoint = _save_and_mirror

trainer.train(max_epochs=EPOCHS, patience=PATIENCE, resume=True)

In [ ]:
# Cell 6: Plot training curves
# Works even if training was cut short — history.json is written every epoch.
import json, matplotlib.pyplot as plt

with open('models/saved/history.json') as f:
    h = json.load(f)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(h['train_loss'], label='Train loss')
ax.plot(h['val_loss'],   label='Val loss')
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE loss')
ax.set_title('Reconstruction loss — pump autoencoder (CWRU real data)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('training_curves.png', dpi=150)
plt.show()
best = min(h['val_loss'])
print(f'Best val loss: {best:.6f} at epoch {h["val_loss"].index(best) + 1}')
print(f'Epochs completed so far: {len(h["val_loss"])}')

In [ ]:
# Cell 7: Calibrate anomaly threshold + evaluate on real fault data
#
# Threshold = 99th percentile of reconstruction error on NORMAL validation windows.
# Then we score the FAULT signal windows and measure precision / recall / AUC.

import sys, numpy as np, torch
sys.path.insert(0, '.')
from models.autoencoder import PumpAutoencoder
from utils.dataset import VibrationDataset
from utils.detector import AnomalyDetector, export_onnx
from torch.utils.data import DataLoader

WINDOW = 512
BATCH  = 128

# Load best weights
model = PumpAutoencoder(input_length=WINDOW, latent_dim=32)
ckpt  = torch.load('models/saved/checkpoint_best.pt', map_location='cuda')
model.load_state_dict(ckpt['model_state'])
print(f'Loaded best checkpoint (val loss: {ckpt["best_val_loss"]:.6f})')

detector = AnomalyDetector(model, device='cuda')

# Calibrate on normal validation windows
normal_signal = np.load('data/raw/normal_signal.npy')
split = int(len(normal_signal) * 0.9)
val_ds     = VibrationDataset(normal_signal[split:], WINDOW, WINDOW // 4)
val_loader = DataLoader(val_ds, batch_size=BATCH, shuffle=False)
detector.calibrate(val_loader, percentile=99.0)
detector.save_threshold('models/saved/threshold.json')

# Evaluate on combined normal + fault windows
fault_signal = np.load('data/raw/fault_signal.npy')

n_normal_eval = min(len(normal_signal) - split, len(fault_signal))
eval_signal   = np.concatenate([normal_signal[split:split + n_normal_eval], fault_signal[:n_normal_eval]])
eval_ds       = VibrationDataset(eval_signal, WINDOW, WINDOW // 4)
eval_loader   = DataLoader(eval_ds, batch_size=BATCH, shuffle=False)

n_normal_windows = len(VibrationDataset(normal_signal[split:split + n_normal_eval], WINDOW, WINDOW // 4))
n_fault_windows  = len(VibrationDataset(fault_signal[:n_normal_eval], WINDOW, WINDOW // 4))
labels = np.array([0] * n_normal_windows + [1] * n_fault_windows)
labels = labels[:len(eval_ds)]

metrics = detector.evaluate(eval_loader, labels)

# Export to ONNX for CPU inference
model.cpu()
export_onnx(model, WINDOW, 'models/saved/pump_autoencoder.onnx')
print('\nAll artifacts ready in models/saved/')

In [ ]:
# Cell 8: Visualise reconstruction — normal vs real fault signal
import torch, numpy as np, matplotlib.pyplot as plt

model = PumpAutoencoder(input_length=512, latent_dim=32)
ckpt  = torch.load('models/saved/checkpoint_best.pt', map_location='cpu')
model.load_state_dict(ckpt['model_state'])
model.eval()

normal_signal = np.load('data/raw/normal_signal.npy')
fault_signal  = np.load('data/raw/fault_signal.npy')

def get_window(sig, start=0):
    w = sig[start:start + 512].astype(np.float32)
    w = (w - w.mean()) / (w.std() + 1e-8)
    return torch.from_numpy(w).unsqueeze(0).unsqueeze(0)

fig, axes = plt.subplots(2, 2, figsize=(14, 6))
for i, (sig, label) in enumerate([(normal_signal, 'Normal (CWRU baseline)'),
                                    (fault_signal,  'Fault (CWRU ball defect)')]):
    x = get_window(sig)
    with torch.no_grad():
        x_hat = model(x)
    orig  = x.squeeze().numpy()
    recon = x_hat.squeeze().numpy()
    err   = float(((orig - recon) ** 2).mean())
    color = 'crimson' if i else 'steelblue'

    axes[i, 0].plot(orig,  lw=0.8, label='Original')
    axes[i, 0].plot(recon, lw=0.8, linestyle='--', label='Reconstructed')
    axes[i, 0].set_title(f'{label}  |  MSE: {err:.5f}')
    axes[i, 0].legend()
    axes[i, 0].grid(True, alpha=0.3)

    residual = np.abs(orig - recon)
    axes[i, 1].fill_between(range(512), residual, alpha=0.7, color=color)
    axes[i, 1].set_title(f'{label}  |  Residual |x - x_hat|')
    axes[i, 1].grid(True, alpha=0.3)

plt.suptitle('Autoencoder reconstruction: normal vs fault bearing vibration (CWRU)', y=1.02)
plt.tight_layout()
plt.savefig('reconstruction_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Cell 9: Download ONNX model + threshold for local CPU inference
from google.colab import files
files.download('models/saved/pump_autoencoder.onnx')
files.download('models/saved/threshold.json')
files.download('training_curves.png')
files.download('reconstruction_comparison.png')
print('Place pump_autoencoder.onnx and threshold.json in models/saved/ locally, then run:')
print('  python infer.py --source data/raw/your_sensor.csv')